# Testing the model
- using 500 rows from Medsynth, not used in training

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install unsloth
!pip install modelscope -U
!pip install evaluate
!pip install hf_transfer

In [ ]:
# The path for the dataset for training 
shuffled_dataset_path = "/kaggle/working/medsynth_preprocessed_shuffled"
base_model_name = "Qwen/Qwen3.5-2B"
trained_adapter_path = ""

In [ ]:
import os

# for kaggle
# import shutil
# if os.path.exists("/kaggle/working/unsloth_compiled_cache"):
#     shutil.rmtree("/kaggle/working/unsloth_compiled_cache")
#     print("Cash deleted.")

# hf_transfer activation (before loading a model)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# optional, not to use modelscope for model loading
os.environ["UNSLOTH_USE_MODELSCOPE"] = "0"
# before import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
import unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorForSeq2Seq
from datasets import load_from_disk

In [ ]:
base_model_name = "Qwen/Qwen3.5-2B"
adapter_path = "/kaggle/input/notebooks/ilovesamir/dial2note-training-2/qwen_9"
max_seq_length = 2048

# 1. base model loading
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. model + LoRA adapter (manual integration)
model.load_adapter(trained_adapter_path)
model = FastLanguageModel.for_inference(model)
print("Model + adapter successful")

# automatic integraion
# unsloth will find the base model according to the adapter config
# But this approach has a problem,
# it loads the parameter in a wrong location
# model, processor = FastLanguageModel.from_pretrained(
#     model_name = adapter_path,
#     max_seq_length = max_seq_length,
#     dtype = None,
#     load_in_4bit = True,
# )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

In [ ]:
# 다시 Dataset 형태로 복구
ds = load_from_disk(shuffled_dataset_path)
total_count = len(ds)
test_ds = ds.select(range(total_count-500, total_count))
print("Test dataset ready")

중복 제거 전: 10238개 -> 중복 제거 후: 10033개
Test dataset ready


In [ ]:
# # A
# instruction = (
#     "Task: Convert the medical dialogue into a professional SOAP note.\n"
#     "Guidelines:\n"
#     "1. STRUCTURE:\n"
#     " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
#     " - Objective: Observable and measurable findings from physical examination.\n"
#     " - Assessment: Clinical diagnosis and medical reasoning.\n"
#     " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
#     "2. CONSTRAINTS:\n"
#     " - Use formal clinical terminology.\n"
#     " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
#     " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
#     " - Start immediately with '**1. Subjective:**'."
# )

# F
# instruction = (
#     "Task: Convert the medical dialogue into a professional SOAP note.\n"
#     "Guidelines:\n"
#     "1. STRUCTURE:\n"
#     " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
#     " - Objective: Observable and measurable findings from physical examination.\n"
#     " - Assessment: Clinical diagnosis and medical reasoning.\n"
#     " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
#     "2. CONSTRAINTS:\n"
#     " - Use formal clinical terminology.\n"
#     " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
#     " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
#     " - Start immediately with '**1. Subjective:**'.\n"
#     " - Use bullet points for high readability."
# )

instruction = (
    "Task: Convert the medical dialogue into a professional SOAP note.\n"
    "Guidelines:\n"
    "1. STRUCTURE:\n"
    " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
    " - Objective: Observable and measurable findings from physical examination.\n"
    " - Assessment: Clinical diagnosis and medical reasoning.\n"
    " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
    "2. CONSTRAINTS:\n"
    " - Use formal clinical terminology.\n"
    " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
    " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
)

example_dialogue = """
[Doctor]: Hello there. What brings you in today?

[Patient]: Hi, doctor. Two days ago, I tripped while going down the stairs, and since then, my right ankle has been really painful and swollen.

[Doctor]: I’m sorry to hear that. Does the pain get worse when you try to walk?

[Patient]: Yes, it’s not too bad when I’m resting, but as soon as I put weight on it or try to walk, the pain is much sharper.

[Doctor]: I see. Let me take a look at the ankle. (After a brief examination) I see some bruising and swelling around the outer ankle bone here. Does it hurt when I press this spot?

[Patient]: Ow! Yes, that really hurts. Do you think it’s broken?

[Doctor]: It’s hard to tell for sure right now. Your vitals—blood pressure and temperature—look normal, but because you have tenderness and swelling over the lateral malleolus, we need to rule out a fracture.

[Patient]: Okay. I haven't had any fever or anything, but the pain is the main issue. What’s the plan?

[Doctor]: First, I’ll prescribe Ibuprofen 400mg for the pain. You can take it every 6 hours as needed. Also, I’m ordering a 3-view X-ray of your right ankle to get a better look at the bone.

[Patient]: That makes sense. Should I come back here after the X-ray?

[Doctor]: Exactly. Once the X-ray results are ready, we’ll review them together and decide on the next steps for your treatment.
"""

example_note = (
    "1. **Subjective:**\n\n"
    "**Chief Complaint (CC):**\n"
    "- Right ankle pain and swelling.\n\n"
    "**History of Present Illness (HPI):**\n"
    "- The patient presents with right ankle pain and swelling following a trip on the stairs 2 days ago. The pain is described as sharp upon weight-bearing or walking and improves with rest. There is no history of fever or similar prior injuries.\n\n"
    "**Review of Systems (ROS):**\n"
    "- Musculoskeletal: Positive for right ankle pain, swelling, and bruising.\n"
    "- General: Negative for fever or chills.\n"
    "- Cardiovascular: No complaints.\n"
    "- Respiratory: No complaints.\n\n"
    "2. **Objective:**\n\n"
    "**Vital Signs:**\n"
    "- BP: 120/80 mmHg\n"
    "- HR: 72 bpm\n"
    "- RR: 18 breaths/min\n"
    "- Temp: 98.6°F\n"
    "- SpO2: 98% on room air\n\n"
    "**Physical Examination:**\n"
    "- Musculoskeletal: Bruising and swelling noted around the outer ankle. Tenderness localized over the right lateral malleolus.\n\n"
    "3. **Assessment:**\n\n"
    "**Diagnosis:**\n"
    "- Acute right ankle sprain. Rule out lateral malleolus fracture due to significant tenderness and swelling.\n\n"
    "4. **Plan:**\n\n"
    "**Medical Management:**\n"
    "- Ibuprofen 400 mg orally every 6 hours as needed for pain.\n\n"
    "**Investigations Ordered:**\n"
    "- X-ray of the right ankle (3 views) to rule out fracture.\n\n"
    "**Follow-Up:**\n"
    "- The patient is instructed to return to the clinic for a review of X-ray results and to determine further treatment steps."
)




def get_predictions(model, tokenizer, dialogue_list, batch_size=8):
    tokenizer.padding_side = 'left'
    all_predictions = []
    few_shot_user = f"Dialogue:\n{example_dialogue}"
    few_shot_assistant = example_note
    anchored_text = "1. **Subjective:**\n\n**Chief Complaint (CC):**\n-"
    for i in range(0, len(dialogue_list), batch_size):
        batch_texts = dialogue_list[i : i + batch_size]
        
        # --- Should be the same as that in training ---
        prompts = []
        for d in batch_texts:
            messages = [
                {"role": "system", "content": instruction}, # 이전에 수정한 상세 지시문
    
                # Few-shot
                {"role": "user", "content": few_shot_user},
                {"role": "assistant", "content": few_shot_assistant},
    
                {"role": "user", "content": f"Dialogue:\n{d}"}
            ]
            # add_generation_prompt=True: automatically produces <|im_start|>assistant\n
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            prompt += anchored_text
            prompts.append(prompt)
        # -----------------------------------------------
        
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id # EOS 명시
            )
        
        input_len = inputs.input_ids.shape[1]
        final_outputs = outputs[:, input_len:]
        
        decoded_outputs = tokenizer.batch_decode(final_outputs, skip_special_tokens=True)
        # all_predictions.extend([p.strip() for p in decoded_outputs])
        for p in decoded_outputs:
          full_note = anchored_text + p
          all_predictions.append(full_note.strip())
        
        
    return all_predictions

In [ ]:
dials = test_ds['Dialogue']
references = test_ds[' Note']
predictions = get_predictions(model, tokenizer, dials, batch_size=32)
print("Finished generation")

Finished generation


In [ ]:
import evaluate

# Loading the metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")

def calculate_all_metrics(predictions, references):
    # ROUGE-1, ROUGE-2, ROUGE-L
    rouge_results = rouge_metric.compute(
        predictions=predictions, 
        references=references, 
        use_stemmer=True
    )
    
    # BLEU
    bleu_results = bleu_metric.compute(
        predictions=predictions, 
        references=references
    )
        
    # METEOR
    meteor_results = meteor_metric.compute(
        predictions=predictions, 
        references=references
    )
    
    print("\n" + "="*40)
    print("📋 Final Performance Metrics")
    print("="*40)
    print(f"🔹 BLEU     : {bleu_results['bleu']:.4f}")
    print(f"🔹 ROUGE-1 : {rouge_results['rouge1']:.4f}")
    print(f"🔹 ROUGE-2 : {rouge_results['rouge2']:.4f}")
    print(f"🔹 ROUGE-L : {rouge_results['rougeL']:.4f}")
    print(f"🔹 METEOR  : {meteor_results['meteor']:.4f}")
    print("="*40)

calculate_all_metrics(predictions, references)

# prediction and reference pair check
clean_predictions = [pred.replace('\\n', '\n') for pred in predictions]
print("Prediction: ")
print(clean_predictions[0])
print("####################################################")
print("####################################################")
print("####################################################")
print("Reference: ")
clean_references = [ref.replace('\\n', '\n') for ref in references]
print(clean_references[0])

import pandas as pd
df = pd.DataFrame({
    "predictions": clean_predictions,
    "references": clean_references
})

# 2. save the result as CSV
df.to_csv("inference_result.csv", index=False, encoding="utf-8-sig")
print("두 리스트가 각각의 컬럼으로 저장되었습니다!")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



📋 Final Performance Metrics
🔹 BLEU     : 0.4149
🔹 ROUGE-1 : 0.6351
🔹 ROUGE-2 : 0.3718
🔹 ROUGE-L : 0.4454
🔹 METEOR  : 0.5549
Prediction: 
#### 1. Subjective

**Chief Complaint (CC):**
- Dyspnea (shortness of breath), fatigue, palpitations, and bilateral leg swelling.

**History of Present Illness (HPI):**
- Patient reports symptoms of dyspnea, fatigue, and palpitations for approximately six months.
- Symptoms are present daily and progressively worsening, particularly with activities requiring exertion such as climbing stairs or walking.
- Palpitations are described as occurring randomly throughout the day, sometimes described as a "skipping beat."
- Leg swelling is noted to be worse by the end of the day and significantly worse at night, with mornings being slightly better.
- No history of wheezing, crackles, nausea, vomiting, abdominal pain, headaches, dizziness, or numbness.
- Patient reports feeling "worn out" but managing well, adhering to previous medical advice.

**Review of Sys